In [ ]:
# Cell 1 — Drive mount + repo clone + symlinks
# Why symlinks: a Colab runtime disconnect wipes /content entirely. data/ and
# artifacts/ must physically live on Drive so a disconnect costs minutes, not a
# full re-download and re-tile.
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/pcb-defect')
(DRIVE / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE / 'artifacts' / 'figures').mkdir(parents=True, exist_ok=True)
(DRIVE / 'mlruns').mkdir(parents=True, exist_ok=True)

%cd /content
!rm -rf /content/pcb-defect
!git clone https://github.com/akshatyuvan/pcb-defect.git
%cd /content/pcb-defect

# Replace the repo's data/ and artifacts/ with symlinks pointing into Drive.
!rm -rf data artifacts
!ln -s {DRIVE}/data data
!ln -s {DRIVE}/artifacts artifacts
!ls -la | grep -E 'data|artifacts'

In [ ]:
# Cell 2 — deps. Torch is preinstalled on Colab with a matching CUDA build.
# Do NOT reinstall it: pip will fetch a mismatched wheel and a long rebuild.
!pip -q install "mlflow>=2.16" opencv-python-headless
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# Cell 3 — fetch DeepPCB into data/raw (a symlink to Drive)
!bash scripts/get_data.sh data/raw

In [ ]:
# Cell 4 — verify BEFORE spending ten minutes tiling
import sys
sys.path.insert(0, '/content/pcb-defect')
from pathlib import Path
from src.data.deeppcb import verify_dataset
verify_dataset(Path('data/raw/PCBData'))

In [ ]:
# Cell 5 — derive the patch-level task from the detection annotations
!python -m src.data.build_patches --raw data/raw/PCBData --out data/patches \
    --assign frac --min-frac 0.25
# good:defect ratio = 8.9:1

In [ ]:
# Cell 6 — THE EYEBALL. Ten patches per class, labels read from the arrays.
# The single most important cell of Day 1. If the class mapping is wrong, every
# number in the next eight days is wrong and nothing will crash to tell you.
import numpy as np, matplotlib.pyplot as plt, json
from src.data.deeppcb import CLASSES

d = np.load('data/patches/train.npz')
X, y = d['X'], d['y']

fig, axes = plt.subplots(len(CLASSES), 10, figsize=(16, 11))
rng = np.random.default_rng(0)
for r, cname in enumerate(CLASSES):
    idx = np.flatnonzero(y == r)
    pick = rng.choice(idx, size=min(10, len(idx)), replace=False)
    for c in range(10):
        ax = axes[r, c]; ax.axis('off')
        if c < len(pick):
            ax.imshow(X[pick[c]], cmap='gray', vmin=0, vmax=255)
    axes[r, 0].axis('on')
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    axes[r, 0].set_ylabel(cname, fontsize=9)
plt.tight_layout()
plt.savefig('artifacts/figures/day1_class_grid.png', dpi=120)
plt.show()

print(json.load(open('data/patches/patch_stats.json'))['splits']['train']['per_class'])

In [ ]:
# Cell 6b — draw each recorded box on the patch it was assigned to.
# If a red rectangle does not sit on the visible defect, either the labelling
# rule or the patch-local coordinate maths is wrong. Both are silent failures
# that would surface as a meaningless Day 4 pointing-game score.
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mp
from src.data.deeppcb import CLASSES

d = np.load('data/patches/train.npz')
X, y = d['X'], d['y']
bp, bloc, bcls, bfrac, bprim = (d['box_patch_idx'], d['box_local'],
                                d['box_class'], d['box_frac'], d['box_primary'])

fig, axes = plt.subplots(6, 6, figsize=(11, 11))
rng = np.random.default_rng(1)
for r in range(6):                       # one row per defect class, ids 1..6
    rows = np.flatnonzero((bcls == r + 1) & bprim)
    pick = rng.choice(rows, size=min(6, len(rows)), replace=False)
    for c in range(6):
        ax = axes[r, c]; ax.axis('off')
        if c >= len(pick):
            continue
        k = pick[c]
        ax.imshow(X[bp[k]], cmap='gray', vmin=0, vmax=255)
        x1, y1, x2, y2 = bloc[k]
        ax.add_patch(mp.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  fill=False, color='red', lw=1.5))
        ax.set_title(f"{CLASSES[bcls[k]]} f={bfrac[k]:.2f}", fontsize=7)
plt.tight_layout()
plt.savefig('artifacts/figures/day1_boxes_on_patches.png', dpi=130)
plt.show()

print("patch label vs box class agreement:",
      (y[bp[bprim]] == bcls[bprim]).mean())   # MUST be exactly 1.0

In [ ]:
# Cell 7 — PCBNet forward pass and parameter count
!python -m src.models.cnn

In [ ]:
# Cell 8 — everything below must be on Drive, not on the ephemeral local disk
!ls -la artifacts artifacts/figures
!ls data/patches